# Preprocessing Pipeline
## Postoperative Complications After Lung Cancer Surgery — DLCAS-2026

---

**Pipeline steps:**
1. Load raw data + sentinel cleaning
2. Search for missing columns (alcohol, CCI)
3. Feature cleaning (age outliers, BMI computation, opnduur unit fix)
4. Charlson Comorbidity Index (CCI)
5. Feature encoding (sex, procedure, staging, access)
6. Drop high-missing and near-zero-variance features
7. Define outcome + train/test split
8. Imputation (fit on train, apply to test — no leakage)
9. Save preprocessed datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import warnings
import os

try:
    import pyreadstat
except ImportError:
    raise ImportError("pip install pyreadstat")

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# ── USER CONFIG ───────────────────────────────────────────────
# NOTE: source registry data is not included in this public repo.
# Paths below are relative placeholders — point them at your own copy.
FILE_PATH  = "../../data/dlca_lung_raw.sav"     # raw DLSA registry export (not in repo)
OUTPUT_DIR = "../../data/lung_processed"        # preprocessed train/test CSVs written here

# Set to True to restrict to primary lung cancer resections only (reggroep == 1)
# Set to False to keep all procedure types (metastasectomy, other)
RESTRICT_TO_LUNG_CANCER = False

# Test set fraction and random seed
TEST_SIZE   = 0.20
RANDOM_SEED = 42

# Outcome column
OUTCOME_COL = 'compl'

# Sentinel values
BINARY_SENTINELS  = [9]
NUMERIC_SENTINELS = [999, 9999, -99, -999, 99.9]

In [ ]:
df, meta = pyreadstat.read_sav(FILE_PATH)
df.columns = [c.lower() for c in df.columns]
meta_labels    = {k.lower(): v for k, v in meta.column_names_to_labels.items()}
meta_val_labels = {k.lower(): v for k, v in meta.variable_value_labels.items()}

print(f"Raw dataset: {df.shape[0]} rows, {df.shape[1]} columns")

# ── Sentinel cleaning ─────────────────────────────────────────
# (same logic as descriptive notebook — run again for clean pipeline)
CORE_COLS = [
    'leeftijd', 'geslacht', 'lengte', 'gewicht',
    'asascore', 'ecog', 'prelongf', 'prelongf1', 'prelongfvo2max',
    'compulmobstr', 'compulmfibr', 'compulmtranspl', 'comap', 'commyo',
    'comcabg', 'comhartfaal', 'comaf', 'comcarritme', 'comhypert',
    'comperif', 'comcva', 'comparalys', 'comdem', 'comnier', 'comdiam',
    'commalig', 'comlever', 'comgiulcus', 'combind', 'comhiv',
    'ctscan', 'pet', 'tumorpetpos', 'tumcentraal',
    'prescoretnm8ct', 'prescoretnm8cn', 'prescoretnm8cm',
    'neoadj', 'inductccrtx', 'inductctx', 'inductrtx', 'inductbiol', 'inductimmuno',
    'reggroep', 'procok', 'toegang', 'zijde', 'sleeve',
    'veraanv', 'verbronc', 'verthor', 'verdiaf', 'verperi', 'vercard', 'vervaat',
    'opnduur', 'compbloed', 'compbloedver', 'transfusie', 'peropcompl',
    OUTCOME_COL
]
CORE_COLS = [c for c in CORE_COLS if c in df.columns]

n_cleaned = 0
for col in CORE_COLS:
    series = pd.to_numeric(df[col], errors='coerce')
    unique_vals = set(series.dropna().unique())
    if unique_vals.issubset({0.0, 1.0, 9.0}):
        n = series.isin(BINARY_SENTINELS).sum()
        if n > 0:
            df[col] = series.replace(BINARY_SENTINELS, np.nan)
            n_cleaned += n
    else:
        n = series.isin(NUMERIC_SENTINELS).sum()
        if n > 0:
            df[col] = series.replace(NUMERIC_SENTINELS, np.nan)
            n_cleaned += n

print(f"Sentinel values cleaned: {n_cleaned}")

In [ ]:
# ── Search for columns not found under expected names ─────────

# 1. Alcohol
print("Searching for alcohol column...")
alcohol_col = None
for c in df.columns:
    if any(kw in c for kw in ['alcohol', 'alc', 'drank', 'drink']):
        n_valid = pd.to_numeric(df[c], errors='coerce').notna().sum()
        print(f"  Found: {c!r}  (N valid = {n_valid})  label: {meta_labels.get(c, '')}")
        if alcohol_col is None:
            alcohol_col = c
if alcohol_col is None:
    print("  Not found under any alcohol/drink keyword.")

# 2. Pre-computed CCI / Charlson
print("\nSearching for CCI / Charlson column...")
cci_col = None
for c in df.columns:
    if any(kw in c for kw in ['cci', 'charlson', 'comorbid']):
        n_valid = pd.to_numeric(df[c], errors='coerce').notna().sum()
        print(f"  Found: {c!r}  (N valid = {n_valid})  label: {meta_labels.get(c, '')}")
        if cci_col is None:
            cci_col = c
if cci_col is None:
    print("  Not found — will compute from individual comorbidity flags.")

# 3. COVID pre-op (alternative names)
print("\nSearching for pre-op COVID column...")
covid_col = None
for c in df.columns:
    if any(kw in c for kw in ['covid', 'cov', 'corona']):
        n_valid = pd.to_numeric(df[c], errors='coerce').notna().sum()
        print(f"  Found: {c!r}  (N valid = {n_valid})  label: {meta_labels.get(c, '')}")
        if covid_col is None:
            covid_col = c
if covid_col is None:
    print("  Not found.")

print(f"\nSummary — alcohol: {alcohol_col}, cci: {cci_col}, covid: {covid_col}")

---
## 3. Feature Cleaning

| Variable | Issue | Fix |
|---|---|---|
| `leeftijd` | 22 values < 18 or > 100 | → NaN |
| `lengte` / `gewicht` | 44% missing | → derive BMI where possible |
| `bmi` | Not in dataset | Compute from lengte + gewicht |
| `opnduur` | In **hours** (header: 'h'); 0 = missing; > 24h = error | 0 → NaN, > 24 → NaN |
| `prescoretnm8ct` | Code 99 = unknown, TX/Tis not ordinal | → NaN for non-stage codes |

In [ ]:
# ── Age ───────────────────────────────────────────────────────
df['leeftijd'] = pd.to_numeric(df['leeftijd'], errors='coerce')
n_bad_age = ((df['leeftijd'] < 18) | (df['leeftijd'] > 100)).sum()
df.loc[(df['leeftijd'] < 18) | (df['leeftijd'] > 100), 'leeftijd'] = np.nan
print(f"Age: {n_bad_age} implausible values → NaN")
print(f"  Valid ages remaining: {df['leeftijd'].notna().sum()}")
print(f"  Range: {df['leeftijd'].min():.0f} – {df['leeftijd'].max():.0f}  Mean: {df['leeftijd'].mean():.1f}")

# ── BMI ───────────────────────────────────────────────────────
gew  = pd.to_numeric(df['gewicht'], errors='coerce')
len_ = pd.to_numeric(df['lengte'],  errors='coerce')
df['bmi'] = np.where(
    (len_ > 100) & (len_ < 250) & (gew > 20) & (gew < 300),
    gew / ((len_ / 100) ** 2),
    np.nan
)
df.loc[(df['bmi'] < 10) | (df['bmi'] > 65), 'bmi'] = np.nan
print(f"\nBMI: {df['bmi'].notna().sum()} valid values")
print(f"  Range: {df['bmi'].min():.1f} – {df['bmi'].max():.1f}  Mean: {df['bmi'].mean():.1f}")

# ── Operation duration (opnduur is in HOURS) ──────────────────
df['opnduur'] = pd.to_numeric(df['opnduur'], errors='coerce')
n_zero = (df['opnduur'] <= 0).sum()
n_over = (df['opnduur'] > 24).sum()
df.loc[df['opnduur'] <= 0,  'opnduur'] = np.nan  # 0h impossible
df.loc[df['opnduur'] > 24,  'opnduur'] = np.nan  # >24h = data error
print(f"\nopnduur (hours): {n_zero} zeros → NaN, {n_over} values >24h → NaN")
print(f"  Valid: {df['opnduur'].notna().sum()}")
print(f"  Range: {df['opnduur'].min():.1f} – {df['opnduur'].max():.1f}  Median: {df['opnduur'].median():.1f}h")

# ── FEV1% range check (valid: 20–150) ─────────────────────────
df['prelongf'] = pd.to_numeric(df['prelongf'], errors='coerce')
n_bad_fev = ((df['prelongf'] < 10) | (df['prelongf'] > 200)).sum()
df.loc[(df['prelongf'] < 10) | (df['prelongf'] > 200), 'prelongf'] = np.nan
print(f"\nprelongf (FEV1%): {n_bad_fev} implausible values → NaN")

# ── DLCO% range check ─────────────────────────────────────────
df['prelongf1'] = pd.to_numeric(df['prelongf1'], errors='coerce')
n_bad_dlco = ((df['prelongf1'] < 10) | (df['prelongf1'] > 200)).sum()
df.loc[(df['prelongf1'] < 10) | (df['prelongf1'] > 200), 'prelongf1'] = np.nan
print(f"prelongf1 (DLCO%): {n_bad_dlco} implausible values → NaN")

In [ ]:
# ── Charlson Comorbidity Index ────────────────────────────────
# If a pre-computed CCI column was found above, use it.
# Otherwise compute from individual comorbidity flags.
# Note: flags are ~75-83% missing — CCI computed here will also be
# missing for those patients. Use with caution.

if cci_col is not None:
    df['cci'] = pd.to_numeric(df[cci_col], errors='coerce')
    print(f"CCI taken from column '{cci_col}'")
    print(f"  Valid: {df['cci'].notna().sum()}  Mean: {df['cci'].mean():.2f}")

else:
    # Charlson weights (simplified mapping to available flags)
    CCI_WEIGHTS = {
        'commyo':       1,   # prior MI
        'comhartfaal':  1,   # heart failure
        'comperif':     1,   # peripheral vascular
        'comcva':       1,   # cerebrovascular
        'comdem':       1,   # dementia
        'compulmobstr': 1,   # COPD
        'combind':      1,   # connective tissue
        'comgiulcus':   1,   # peptic ulcer
        'comlever':     1,   # mild liver disease
        'comdiam':      1,   # diabetes (use 1; registry may not distinguish complicated)
        'comparalys':   2,   # hemiplegia
        'comnier':      2,   # renal disease
        'commalig':     2,   # malignancy
        'comhiv':       6,   # HIV/AIDS
    }

    cci_present = {col: w for col, w in CCI_WEIGHTS.items() if col in df.columns}
    cci_df = df[list(cci_present.keys())].apply(pd.to_numeric, errors='coerce')

    # CCI is NaN if ALL components are missing (patient has no comorbidity data)
    # If at least one flag is recorded, treat missing flags as 0 (not present)
    n_recorded = cci_df.notna().sum(axis=1)
    cci_filled = cci_df.fillna(0)
    df['cci'] = cci_filled.mul(pd.Series(cci_present)).sum(axis=1)
    df.loc[n_recorded == 0, 'cci'] = np.nan  # fully missing → NaN

    print(f"CCI computed from {len(cci_present)} comorbidity flags.")
    print(f"  Valid (at least 1 flag recorded): {df['cci'].notna().sum()}")
    print(f"  Missing (no comorbidity data):    {df['cci'].isna().sum()}")
    print(f"  Distribution:")
    print(df['cci'].value_counts(dropna=False).sort_index().to_string())

---
## 4. Feature Encoding

| Variable | Original | Encoding |
|---|---|---|
| `geslacht` | 1=Man, 2=Vrouw | Binary: 0=Man, 1=Vrouw |
| `toegang` | 10 codes | Simplified: 0=Open, 1=MIS, 2=Conversion |
| `procok` | 1=Pneumonectomy, 2=Bilob, 3=Lob | Ordinal by extent: 0=Lob, 1=Bilob, 2=Pneumo |
| `zijde` | 0=Links, 1=Rechts | Binary: 0=Left, 1=Right |
| `sleeve` | 0–3 | Ordinal: 0=None, 1=Bronchial, 2=Arterial, 3=Double |
| `compbloed` | 0–3 | Ordinal: 0=0-500cc, 1=500-1000, 2=1000-2000, 3=>2000 |
| `asascore` | 1–4 | Ordinal (kept as-is) |
| `ecog` | 0–4 | Ordinal (kept as-is) |
| `prescoretnm8ct` | 11 codes | Ordinal T1→T4; TX/Tis/unknown → NaN |
| `prescoretnm8cn` | N0–N3 + unknown | Ordinal 0–3; unknown → NaN |
| `prescoretnm8cm` | M0/M1a/b/c | Binary: 0=M0, 1=any M1 |
| `reggroep` | 3 groups | One-hot (ref: lung cancer resection) |

In [ ]:
# ── Sex ───────────────────────────────────────────────────────
df['sex_female'] = pd.to_numeric(df['geslacht'], errors='coerce').map({1.0: 0, 2.0: 1})

# ── Surgical access: simplify to 3 categories ─────────────────
OPEN_CODES = {1.0, 2.0}            # thoracotomie, sternotomie
MIS_CODES  = {3.0, 5.0, 7.0}       # VATS, RATS, UVATS
CONV_CODES = {4.0, 6.0, 8.0, 9.0}  # any conversion

def encode_toegang(val):
    if pd.isna(val): return np.nan
    if val in OPEN_CODES: return 0
    if val in MIS_CODES:  return 1
    if val in CONV_CODES: return 2
    return np.nan  # code 77 = anders

df['toegang_cat'] = pd.to_numeric(df['toegang'], errors='coerce').apply(encode_toegang)
print("toegang_cat distribution:")
print(df['toegang_cat'].value_counts(dropna=False).rename({0:'Open', 1:'MIS', 2:'Conversion', np.nan:'Missing'}))

# ── Procedure extent: ordinal ──────────────────────────────────
# 0 = lobectomy (standard), 1 = bilobectomy, 2 = pneumonectomy
df['procok_ord'] = pd.to_numeric(df['procok'], errors='coerce').map({3.0: 0, 2.0: 1, 1.0: 2})
print("\nprocok_ord distribution:")
print(df['procok_ord'].value_counts(dropna=False).rename({0:'Lobectomy', 1:'Bilobectomy', 2:'Pneumonectomy', np.nan:'Missing/Other'}))

# ── Side ──────────────────────────────────────────────────────
# 0=Links→0, 1=Rechts→1, 2=Beiderzijds→NaN (bilateral=unusual), 3=Mediastinaal→NaN
df['zijde_rechts'] = pd.to_numeric(df['zijde'], errors='coerce').map({0.0: 0, 1.0: 1})

# ── Sleeve resection ordinal ──────────────────────────────────
df['sleeve_ord'] = pd.to_numeric(df['sleeve'], errors='coerce').map({0.0: 0, 1.0: 1, 2.0: 2, 3.0: 3})

# ── Blood loss ordinal ────────────────────────────────────────
df['compbloed_ord'] = pd.to_numeric(df['compbloed'], errors='coerce').map({0.0: 0, 1.0: 1, 2.0: 2, 3.0: 3})

# ── ASA + ECOG (already ordinal, just ensure numeric) ─────────
df['asascore'] = pd.to_numeric(df['asascore'], errors='coerce')
df['ecog']     = pd.to_numeric(df['ecog'],     errors='coerce')

print("\nEncoding complete: sex_female, toegang_cat, procok_ord, zijde_rechts, sleeve_ord, compbloed_ord")

In [ ]:
# ── TNM8 encoding ─────────────────────────────────────────────
# T stage: map SPSS codes to clinical T stage (ordinal 1–7)
# SPSS codes from value labels:
#   3=T1a(mi), 4=T1a, 5=T1b, 6=T1c, 7=T2a, 8=T2b, 9=T3, 10=T4
#   0=T0, 1=TX, 2=Tis, 99=unknown → NaN
T_MAP = {
    3.0:  1,   # T1a(mi)
    4.0:  2,   # T1a
    5.0:  3,   # T1b
    6.0:  4,   # T1c
    7.0:  5,   # T2a
    8.0:  6,   # T2b
    9.0:  7,   # T3
    10.0: 8,   # T4
    # 0=T0, 1=TX, 2=Tis, 99.0=unknown → not mapped → NaN
}
df['tnm_t'] = pd.to_numeric(df['prescoretnm8ct'], errors='coerce').map(T_MAP)

# N stage: N0=0, N1=1, N2=2, N3=3; Nx/unknown → NaN
N_MAP = {0.0: 0, 1.0: 1, 2.0: 2, 3.0: 3}
df['tnm_n'] = pd.to_numeric(df['prescoretnm8cn'], errors='coerce').map(N_MAP)

# M stage: binary  M0=0,  M1a/b/c=1;  unknown → NaN
m_series = pd.to_numeric(df['prescoretnm8cm'], errors='coerce')
df['tnm_m'] = m_series.map({0.0: 0, 1.0: 1, 2.0: 1, 3.0: 1})  # 9=unknown stays NaN

print("TNM8 encoding:")
print(f"  tnm_t: {df['tnm_t'].notna().sum()} valid  ({df['tnm_t'].isna().mean()*100:.1f}% missing)")
print(f"  tnm_n: {df['tnm_n'].notna().sum()} valid  ({df['tnm_n'].isna().mean()*100:.1f}% missing)")
print(f"  tnm_m: {df['tnm_m'].notna().sum()} valid  ({df['tnm_m'].isna().mean()*100:.1f}% missing)")

print("\nT stage distribution:")
t_labels = {1:'T1a(mi)', 2:'T1a', 3:'T1b', 4:'T1c', 5:'T2a', 6:'T2b', 7:'T3', 8:'T4'}
print(df['tnm_t'].value_counts(dropna=False).rename(t_labels))

print("\nN stage distribution:")
print(df['tnm_n'].value_counts(dropna=False))

print("\nM stage distribution:")
print(df['tnm_m'].value_counts(dropna=False))

In [ ]:
# ── Procedure group (reggroep) one-hot ────────────────────────
# 1=lung cancer resection (reference), 3=metastasectomy, 4=other
rg = pd.to_numeric(df['reggroep'], errors='coerce')
df['rg_metastasectomy'] = (rg == 3).astype(float)
df['rg_other']          = (rg == 4).astype(float)
# rg == 1 is the reference category (no separate column needed)

print("reggroep encoding:")
print(rg.value_counts(dropna=False))

if RESTRICT_TO_LUNG_CANCER:
    n_before = len(df)
    df = df[rg == 1].copy()
    print(f"\nRestricted to lung cancer resections: {len(df)} / {n_before} patients kept")
else:
    print("\nAll procedure types kept (RESTRICT_TO_LUNG_CANCER = False)")

# alcohol found as 'alchohol' but dropped — 95.3% missing, too sparse to include
print(f"\nalcohol column ('{alcohol_col}'): found but excluded — {95.3}% missing")

---
## 5. Final Feature Set

Variables dropped:
- `roken` (95% missing)
- All individual comorbidity flags (>72% missing) → replaced by CCI
- All `ver*` extension procedures (>89% missing)
- Neoadjuvant treatment variables (>50% missing)
- `prelongfvo2max` (73% missing)
- `compbloedver` (64% missing; categorical `compbloed` is available)
- `ctscan` (99.5% positive — near-zero variance)
- `tumorpetpos` (96.4% positive — near-zero variance)
- Raw `prescoretnm8*` (replaced by `tnm_t/n/m`)
- Raw `toegang`, `procok`, `zijde`, `geslacht`, `sleeve`, `compbloed` (replaced by encoded versions)

In [ ]:
# Build final feature list
FINAL_FEATURES = [
    # Demographics & body composition
    'leeftijd',       # age (years)
    'sex_female',     # 0=male, 1=female
    'bmi',            # kg/m2 (computed)
    # Functional status
    'asascore',       # ordinal 1-4
    'ecog',           # ordinal 0-4
    'prelongf',       # FEV1% predicted
    'prelongf1',      # DLCO% predicted
    # Comorbidity burden
    'cci',            # Charlson Comorbidity Index
    # Imaging / staging
    'pet',            # PET performed (binary)
    'tumcentraal',    # central tumor (binary)
    'tnm_t',          # clinical T stage ordinal
    'tnm_n',          # clinical N stage ordinal
    'tnm_m',          # M stage binary
    # Procedure
    'procok_ord',     # 0=lobectomy, 1=bilobectomy, 2=pneumonectomy
    'toegang_cat',    # 0=open, 1=MIS, 2=conversion
    'zijde_rechts',   # 0=left, 1=right
    'sleeve_ord',     # 0=none ... 3=double sleeve
    'veraanv',        # additional resection (binary)
    'compbloed_ord',  # intraop blood loss category 0-3
    'transfusie',     # transfusion given (binary)
    'opnduur',        # operation duration (hours)
    # Procedure type (reggroep one-hot; ref = lung cancer resection)
    'rg_metastasectomy',
    'rg_other',
    # Excluded: alcohol (95.3% missing — dropped, not imputed)
    # Excluded: covid (not found in dataset)
]

# Keep only columns that actually exist in the dataframe
FINAL_FEATURES = [f for f in FINAL_FEATURES if f in df.columns]

print(f"Final feature set: {len(FINAL_FEATURES)} variables")
print("\nMissing % per feature:")
miss_summary = pd.DataFrame({
    'Feature':   FINAL_FEATURES,
    'N present': [df[f].notna().sum() for f in FINAL_FEATURES],
    '% missing': [round(df[f].isna().mean()*100, 1) for f in FINAL_FEATURES]
})
print(miss_summary.to_string(index=False))

In [ ]:
# ── Outcome definition ────────────────────────────────────────
y_raw = pd.to_numeric(df[OUTCOME_COL], errors='coerce')

# Keep only rows with a valid outcome
valid_idx = y_raw.notna()
X_full = df.loc[valid_idx, FINAL_FEATURES].copy()
y_full = y_raw[valid_idx].astype(int)

print(f"Patients with valid outcome: {len(y_full)} / {len(df)}")
print(f"\nClass balance (compl):")
vc = y_full.value_counts()
print(f"  0 (no complication): {vc.get(0, 0)}  ({vc.get(0,0)/len(y_full)*100:.1f}%)")
print(f"  1 (complication):    {vc.get(1, 0)}  ({vc.get(1,0)/len(y_full)*100:.1f}%)")

# Visual
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['No complication (0)', 'Complication (1)'],
       [vc.get(0,0), vc.get(1,0)],
       color=['#88BBDD', '#E07050'], edgecolor='white')
ax.set_ylabel('N patients')
ax.set_title('Outcome Distribution', fontweight='bold')
for i, v in enumerate([vc.get(0,0), vc.get(1,0)]):
    ax.text(i, v + 5, str(v), ha='center', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Stratified train / test split ─────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full,
    test_size=TEST_SIZE,
    stratify=y_full,
    random_state=RANDOM_SEED
)

print(f"Train: {len(X_train)} patients  ({y_train.mean()*100:.1f}% complications)")
print(f"Test:  {len(X_test)}  patients  ({y_test.mean()*100:.1f}% complications)")

In [ ]:
# ── Imputation: fit on TRAIN only, apply to both ──────────────
# This prevents any information from the test set leaking into
# the imputed values used during model training.

CONTINUOUS_FEATS = [
    'leeftijd', 'bmi', 'prelongf', 'prelongf1', 'opnduur', 'cci'
]
CATEGORICAL_FEATS = [f for f in FINAL_FEATURES if f not in CONTINUOUS_FEATS]

# Filter to only those present
cont_present = [f for f in CONTINUOUS_FEATS if f in FINAL_FEATURES]
cat_present  = [f for f in CATEGORICAL_FEATS if f in FINAL_FEATURES]

med_imputer  = SimpleImputer(strategy='median')
mode_imputer = SimpleImputer(strategy='most_frequent')

# Fit + transform train
X_train_cont = pd.DataFrame(
    med_imputer.fit_transform(X_train[cont_present]),
    columns=cont_present, index=X_train.index
)
X_train_cat = pd.DataFrame(
    mode_imputer.fit_transform(X_train[cat_present]),
    columns=cat_present, index=X_train.index
)
X_train_imp = pd.concat([X_train_cont, X_train_cat], axis=1)[FINAL_FEATURES]

# Transform test (using train's fitted imputers)
X_test_cont = pd.DataFrame(
    med_imputer.transform(X_test[cont_present]),
    columns=cont_present, index=X_test.index
)
X_test_cat = pd.DataFrame(
    mode_imputer.transform(X_test[cat_present]),
    columns=cat_present, index=X_test.index
)
X_test_imp = pd.concat([X_test_cont, X_test_cat], axis=1)[FINAL_FEATURES]

print("Imputation complete (no leakage from test to train).")
print(f"\nImputed values per feature (train set):")
for col in FINAL_FEATURES:
    n_was_missing = X_train[col].isna().sum()
    if n_was_missing > 0:
        strategy = 'median' if col in cont_present else 'mode'
        imputed_val = X_train_imp[col].iloc[0] if col in cat_present else med_imputer.statistics_[cont_present.index(col)]
        print(f"  {col:<25} {n_was_missing:>5} missing → {strategy} imputed")

# Verify no NaN remains
assert X_train_imp.isna().sum().sum() == 0, "NaN in train after imputation!"
assert X_test_imp.isna().sum().sum()  == 0, "NaN in test after imputation!"
print("\nVerification passed: zero NaN values in train and test sets.")

In [ ]:
# ── Post-imputation feature overview ─────────────────────────
print("FINAL FEATURE SUMMARY (train set, post-imputation)")
print("=" * 65)
for col in FINAL_FEATURES:
    s = X_train_imp[col]
    n_uniq = s.nunique()
    if n_uniq <= 5:
        dist = s.value_counts().sort_index().to_dict()
        print(f"  {col:<25} categories: {dist}")
    else:
        print(f"  {col:<25} mean={s.mean():.2f}  median={s.median():.2f}  "
              f"std={s.std():.2f}  min={s.min():.1f}  max={s.max():.1f}")

In [ ]:
# ── Correlation heatmap (train set) ───────────────────────────
corr = X_train_imp.corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size': 7}, linewidths=0.3)
ax.set_title('Feature Correlation Matrix (train set)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print("\nHighly correlated pairs |r| > 0.70 (potential multicollinearity):")
found = False
for i in range(len(corr.columns)):
    for j in range(i):
        r = corr.iloc[i, j]
        if abs(r) > 0.70:
            print(f"  {corr.columns[i]:<25} <-> {corr.columns[j]:<25}  r = {r:.2f}")
            found = True
if not found:
    print("  None above threshold.")

In [ ]:
# ── Save preprocessed datasets ────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_train_imp.to_csv(os.path.join(OUTPUT_DIR, 'X_train.csv'), index=False)
X_test_imp.to_csv( os.path.join(OUTPUT_DIR, 'X_test.csv'),  index=False)
y_train.to_csv(    os.path.join(OUTPUT_DIR, 'y_train.csv'),  index=False, header=True)
y_test.to_csv(     os.path.join(OUTPUT_DIR, 'y_test.csv'),   index=False, header=True)

# Also save full preprocessed (for exploratory use)
X_full_imp = pd.concat([X_train_imp, X_test_imp]).sort_index()
X_full_imp['compl'] = y_full
X_full_imp.to_csv(os.path.join(OUTPUT_DIR, 'lung_preprocessed_full.csv'), index=False)

print(f"Saved to: {OUTPUT_DIR}")
print(f"  X_train.csv   — {X_train_imp.shape}")
print(f"  X_test.csv    — {X_test_imp.shape}")
print(f"  y_train.csv   — {y_train.shape}")
print(f"  y_test.csv    — {y_test.shape}")
print(f"  lung_preprocessed_full.csv — {X_full_imp.shape}")

---
## Summary & Decisions Log

### What was done

| Step | Decision |
|---|---|
| Sentinel cleaning | 9 → NaN (binary), 999/-99 → NaN (numeric) |
| Age outliers | Values <18 or >100 → NaN, then median imputed |
| BMI | Computed from lengte/gewicht; implausible values (<10, >65) → NaN |
| opnduur | Unit = hours; 0h → NaN; >24h (data errors) → NaN |
| FEV1% / DLCO% | Values outside 10–200 range → NaN |
| Comorbidities | Individual flags (>72% missing) replaced by CCI |
| `toegang` | Simplified to Open / MIS / Conversion |
| `procok` | Ordinal by extent: Lobectomy=0, Bilob=1, Pneumonectomy=2 |
| TNM | Ordinal T stage, N stage; M stage binary; TX/Tis/unknown → NaN |
| Dropped variables | `roken`, `ctscan`, `tumorpetpos`, `ver*`, neoadjuvant, `compbloedver`, `prelongfvo2max` |
| Imputation | Median for continuous, mode for categorical — fit on train only |
| Split | 80/20 stratified on `compl` |

### Known limitations
- **CCI is ~75% missing** if not pre-computed in the registry — confirm with data manager
- **BMI is ~44% missing** — median imputation introduces noise; sensitivity analysis recommended
- **Neoadjuvant treatment** is entirely missing (>50%) — cannot be used as predictor
- **Complication subtypes** (airway, cardiac, etc.) are mostly incomplete — only `compl` (any) is reliable as outcome

### Next step → modeling notebook
1. Logistic Regression (LASSO) — feature selection
2. XGBoost — non-linear effects + SHAP feature importance
3. Stratified 5-fold cross-validation on train set
4. Final evaluation on held-out test set (AUC-ROC, calibration, Brier score)